# Inductive biases

Inductive bias is the set of assumptions a model makes to generalize beyond the training data. Good biases match the structure and symmetries of your data, yielding better sample efficiency and robustness. Here are few examples of inductive biases to consider:


| Domain / Data Type                 | Key Property / Symmetry                        | Typical NN                     | Inductive bias                                                                                                     |
| ---------------------------------- | ---------------------------------------------- | ---------------------------------- | ----------------------------------------------------------------------------------------------------------------------- |
| Images                             | Local stationarity; translation symmetry; pixel-to-pixel correlations       | **CNN**                            | Captures repeating local patterns;  translation equivariance. |
| Time series          | Causality; temporal order; long correlations | **RNN / GRU / LSTM**               | Recurrent state summarizes history; gating mitigates vanishing/exploding gradients; data-efficient.                     |
|  |                 | **Transformer**           | Self-attention models long dependencies in parallel; order via positional encodings.                                    |
| Graphs                             | Permutation symmetry; relational locality      | **GNN** | Neighbor aggregation is permutation-invariant and respects topology; scalable via sampling.                             |



# Convolutional neural networks





## Convolutional layer

### Convolution operation

We start by introducing the convolution operation, i.e. the operation that is performed to compute the activations of a convolutional layer. For simplifity, we will consider here a 2D convolution (i.e. inputs are always 2D images / tensors). The convolution has two input matrices: the input matrix $I$ of dimension $(d_x, d_y)$ and the *convolution kernel* or *filter* $K$ of dimension $(h,w)$. In the simplest case, we compute the convolved image, $I∗K$, by overlaying the kernel on top of the image in all possible ways, and recording the sum of element-wise products between the image and the kernel:

\begin{equation}
(I * K)_{xy} = \sum_{i=1}^{h} \sum_{j=1}^{w} K_{ij} \, I_{x+i-1,\,y+j-1}
\end{equation}

Here is a visual representation of the previous operation:

::: {#fig-cnn}
<img src="../../figures/cnn_ibm.png" width="60%"/>

Convolution operation. Credit: [IBM](https://www.ibm.com/think/topics/convolutional-neural-networks).
:::

It is important to see that a convolution effectively changes the dimension of our input matrix. In particular, for the case above, the resulting matrix is shrunk from $(d_x\times d_y)$ to $(d_x - w + 1\times d_y - h + 1)$. While this may be a problem, we will show some methods to avoid this shrinkage.


By looking at the above equation, we can see that we are just performing a linear operation, hence not fulfilling the constraints needed to construct a universal approximator. We need to introduce now a non-linear function $\sigma$, which is typically applied per element to the previous matrix. With such, a convolutional layer is defined as:

\begin{equation}
\textrm{conv}(I, K) = \sigma((I * K)_{xy} + b)
\end{equation}

where we also introduced a bias $b$, in this case a matrix of same dimension as the output matrix $(I * K)_{xy}$. Since all we are doing here is addition and scaling of the input pixels, the parameters of the kernels $K_{ij}$ and biases $b_{ij}$ may be learned from a given training dataset via gradient descent, exactly as we did in previous neural networks. These are the trainable *parameters* of a convolutional neural network.


### Convolutional layers hyperparameters
From the previous core operation arise a few implementation choices:

- **Kernel size:** this refers to the width $w$ and height $h$ of the kernel. A larger kernel size allows the convolution to capture broader, more global correlations in the input, while a smaller kernel focuses on fine-grained, local patterns. Thus, kernel size determines the receptive field and the spatial scale of features the network can learn.
- **Depth or channels:** channels refer to a new third dimension of the 2D matrices used above. For the input image, this relates for instance to the color channels (hence depth 3). It also relates to the *number of kernels*. Let's see an example with the following network:

::: {#fig-cnn-channels}
<img src="../../figures/cnn_diagram.svg" width="60%"/>

Convolution operation with channels. 
:::
    
    Let's focus on the third dimension only here and go step by step:
    1. The input has a single channel, hence dimension 1.
    2. We now apply a convolution with a kernel size $(5\times5)$. But instead of considering a single kernel, I will stack 16 of them. Importantly, these kernels have their own training parameters and are independent of each other. Because I have 16 kernels, applying the convolution operation above leads to 16 different matrices, which we stacked for an output matrix of size $(13\times13\times16)$.
    3.  We now apply again a convolutional layer, now with filters of size $(3\times3)$. Importantly, the third dimension of the input matrix must coincide with that of the kernel. So actually, our kernels are now three dimensional tensors of size $(3\times3\times16)$. In general, given an input matrix $I_{d_xd_yc}$ and a kernel $K_{whc}$, we are performing:
       \begin{equation}
        (I * K)_{xy} = \sum_{i=1}^{h} \sum_{j=1}^{w} \sum_{k=1}^{c} K_{ijk} \, I_{x+i-1,\,y+j-1,k}
        \end{equation}
       In the following picture we can see this operation:
    
    
::: {#fig-cnn-channels-simp}
<img src="../../figures/cnn_channels.svg"  width="40%"/>

Convolution operation with channels. Credit: [d2l.ai](https://d2l.ai/chapter_convolutional-neural-networks/channels.html)
:::

- **Stride:** refers to the displacement we make when convolving the images. Until now, we have considered a stride of one, but in many cases we may consider a bigger one, as shown here:

::: {#fig-cnn-dims}
<img src="../../figures/cnn_stride.svg"  width="30%"/>

Stride in convolutions
:::

- **Padding:** One problem of the convolution operation is that it effectively reduces the size of the output matrix. This also introduces a disparity on how much each element of the input matrix is used when performing the convolution. For instance, the corner elements are only used once, hence barely contributing to the output. To solve both problems, we consider padding: we add extra pixels around the boundary of our input image, thus increasing the effective size of the image. Typically, we set the values of the extra pixels to zero, although this is also sometimes a key hyperparameter. The following image showcases its effect:

::: {#fig-cnn-dims}
<img src="../../figures/cnn_padding.svg"  width="60%"/>

Padding in convolutions
:::
  As we can see, our input matrix was of dimension $(3\times3)$. From the example in @#fig-cnn-channels, we see that without padding, the output dimension would have been reduced to $(2\times2)$. In contrast, our output matrix has now increased dimension, allowing us to create deeper convolutional neural networks without worrying about reducing the dimension!
  The value of the padding can be set at will, but should not be smaller than the kernel size, as if not we will have elements of the output matrix that are just zero.

- **Output dimensions of the convolution:** depending on the parameters above, the output dimension may vary drastically. It is hence important to keep track of the dimensions after each layer. Considering again an input matrix $I$ of dimension $(d_x, d_y)$ and the filter $K$ of dimension $(h,w)$, a stride $s$ and a padding of $p$, the output dimension is given by:

\begin{aligned}
d_x' &= \left\lfloor \frac{d_x - w + 2p}{s} \right\rfloor + 1, \\
d_y' &= \left\lfloor \frac{d_y - h + 2p}{s} \right\rfloor + 1.
\end{aligned}
    
 Typically, `pytorch` and other libraries will take care of this for us, but during debugging it's always helpful to keep this in mind!

### Convolution operation with `pytorch`
Let's put in practice the previous concepts with `pytorch`. We first explore how to do a simple convolution: 

In [ ]:
import torch

dx, dy = 6, 8
# IMPORTANT: the first dimension corresponds to the number of channels and must be always specified, even when equal to 1
I = torch.rand((1, dx,dy))

h, w = 3, 4
conv = torch.nn.Conv2d(in_channels = 1, # We are considering here a 2D matrix, so just one channel
                    out_channels = 1, # This refers to the number of kernels, let's keep at 1 and comeback to it later
                    kernel_size = (h,w))

# Now we perform the convolution of I w.r.t. m. m behaves as a layer, so we can just do:
O = conv(I)

# Let's look at the output and its shape
O, O.shape

(tensor([[[-0.5958, -0.3135, -0.3103, -0.0812, -0.5546],
          [-0.4480, -0.4272, -0.3126, -0.2641, -0.0945],
          [-0.2474, -0.4487, -0.3146, -0.4519, -0.1672],
          [-0.3792, -0.0122, -0.1815, -0.6858, -0.1872]]],
        grad_fn=<SqueezeBackward1>),
 torch.Size([1, 4, 5]))

In this previous example we kept the number of kernels (`out_channels` input variable) at one. We can set this at will. Moreover, `pytorch` we take into account the input dimensionality when creating the kernels. Let's for instance reproduce the example we saw in @fig-cnn-channels

In [ ]:
I = torch.rand((1, 28, 28))

conv1 = torch.nn.Conv2d(in_channels = 1, # the input is still 1
                        out_channels = 16, # we now consider 16 kernels
                        kernel_size = (5, 5))

conv2 = torch.nn.Conv2d(in_channels = 16, # because the previous out is 16, we consider that as input
                        out_channels = 2, # we now consider 2 kernels
                        kernel_size = (3, 3))

# Now we can compute the convolutions:

out1 = conv1(I)
print(f'Dimension after 1st conv: {out1.shape}')

out2 = conv2(out1)
print(f'Dimension after 2nd conv: {out2.shape}')

Dimension after 1st conv: torch.Size([16, 24, 24])
Dimension after 2nd conv: torch.Size([2, 22, 22])


In the same way, we can introduce padding and stride easily:

In [ ]:
I = torch.rand((1, 28, 28))

conv_pad_str = torch.nn.Conv2d(in_channels = 1,
                                 out_channels = 16,
                                 kernel_size = (5, 5),
                                 stride = 3,
                                 padding = 4
                                )
conv_pad_str(I).shape

torch.Size([16, 11, 11])

You can play with the numbers above to get an idea on how these parameters interplay.

### Pooling operation
Pooling layers reduce the spatial dimensions of feature maps by summarizing local regions, typically using operations like max or average. This downsampling helps make the network more computationally efficient and robust to small spatial translations or distortions. Let's see an example: 

::: {#fig-cnn-maxpool}
<img src="../../figures/cnn_maxpool.png"  width="70%"/>

Pooling layer example. The pooling above considers a stride of 2.
:::

Pooling layers are defined by the size $(f_x\times f_y)$ of their feature maps, i.e. the size of a region from which a result is pooled, and the stride $s$, which works just for convolutional layers. Consider again an input matrix of dimensions $(d_x\times d_y)$, the output size is given by

\begin{aligned}
d_x' &= \left\lfloor \frac{d_x - f_x}{s} \right\rfloor + 1, \\
d_y' &= \left\lfloor \frac{d_y - f_y}{s} \right\rfloor + 1,
\end{aligned}

In @fig-cnn-maxpool we considered a max pooling operation, although other pooling operations exist: 
- Max pooling: computes the maximum of the feature map
- Average pooling: computes the average of the feature map
- Global Pooling: this is an special operation, which reduces each channel of the feature map to a single value. For instance, if our matrix is of size $(d_x\times d_y\times d_c)$, performing gobal pooling would transform it to a matrix $(1\times 1\times d_c)$. It is analog to considering a a feature map of size $(d_x\times d_y)$. Again, both the max or average operations can be considered.

This type of layer is particularly useful at then end of our convolutional neural network, to help us flatten the tensors in a robust way, able to backward pass gradients. Let's first see an example of max and average pooling: 

In [ ]:
I = torch.rand((1, 6,6))

max_pool = torch.nn.MaxPool2d(3, stride=2)

avg_pool = torch.nn.AvgPool2d(3, stride=2)

max_pool(I), avg_pool(I)

(tensor([[[0.9925, 0.8997],
          [0.8536, 0.8997]]]),
 tensor([[[0.6202, 0.4331],
          [0.5649, 0.4556]]]))

Pytorch does not have a built-in global pooling layer, but we can built it ourselves by considering the dimensions of the input matrix:

In [ ]:
I = torch.rand((15,6,6)) # Let's consider few channels

global_pool = torch.nn.MaxPool2d(I.shape[-2:]) # We consider a feature map with size the last two dimensions of input matrix

global_pool(I).shape

torch.Size([15, 1, 1])

## Architectures
We will now explore few different architectures we can build based on convolutional layers.

### Creating a CNN with pytorch
We will start by using `pytorch`'s modules to create a CNN. It works just as with the feed forward layers. We will consider that our input images are grayscale, i.e. a single input channel. Note that because of the nature of convolutional layer, we do not care about the other dimensions, and our network would in principle work fine with any one channel image. 

In [ ]:
import torch.nn.functional as F

class ConvNN(torch.nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = torch.nn.Conv2d(in_channels = 1, # we get greyscale images (one channels)
                                     out_channels = 16, # we now consider 16 kernels
                                     kernel_size = (5, 5))
        
        self.conv2 = torch.nn.Conv2d(in_channels = 16, # because the previous out is 16, we consider that as input
                                     out_channels = 12, # we now consider 12 kernels
                                     kernel_size = (3, 3))
        
        # Let's also add a final linear layer to get the desider output shape, let's say for this example 5.
        # Because our last conv layer has 12 channels, we can later implement a global pooling and end up
        # with a tensor of single dimension and shape 12. This gives us the shape our linear layer should have:
        self.linear_1 = torch.nn.Linear(12, 5)
        
    def forward(self, x):
        z = self.conv1(x)
        x = F.relu(z)
        z = self.conv2(x)
        x = F.relu(z)
        # Now we implement the global pooling. Because we may not know the dimensions of the input matrix,
        # we calculate them on the flight
        x = torch.nn.MaxPool2d(x.shape[-2:])(x)   
        # We can get rid of the "spurious" dimensions (those equal to one) using the squeeze function
        x = x.squeeze()
        
        z = self.linear_1(x)
        # we can decide or not to perform an activation function here :)
        return z 
    

In [ ]:
# Let's define our model
model = ConvNN()

# And test it with a batch of 64 images of dimension (32x32)
batch = torch.rand((64, # batch size
                    1, # num channels
                    32, # dx
                    32 #dy
                   ))

out = model(batch)
out.shape

torch.Size([64, 5])

::: {.callout-tip}
## Exercise

Create a CNN and train it to classify MNIST as we did in the section "Deep learning à la torch" in [this notebook](02_nn_with_pytorch.ipynb). Can you improve those results? Plot the accuracy and check that it is higher than in the feed forward network.

**Bonus:** when comparing NN, it is important to keep the number of parameters equal. This makes the comparison fairer, as the advantage will not arise from having a more powerful network (more parameters) but rather on its inductive bias. Based on the maths we learned through this notebook and in previous, create two networks, one based on linear layers and one based on convolutional layers, with (approximately) the same number of parameters. Then retrain on MNIST and compare.
:::